<a href="https://colab.research.google.com/github/nithin12342/phase2/blob/main/ml_pipeline/h5_omnifusion/notebooks/Phase7_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Phase 7 Training Notebook (Fine-Tuning from Phase 6)

## Key Fixes in Phase 7:
- **Stratified test split** - Ensures test set has proportional class distribution
- **Label smoothing 0.1** - Reduces overfitting (was 0.02)
- **Fine-tuning from Phase 6** - Resume from Phase 6 checkpoints instead of scratch
- **focal_alpha 0.75** - Better minority class handling (was 0.65)
- **mixup_alpha 0.3** - Less aggressive augmentation (was 0.4)
- **audio_dropout 0.20** - Reduced dropout (was 0.35)

## Expected Results:
| Metric | Phase 6 | Phase 7 Target |
|--------|---------|----------------|
| **Ensemble F1** | 0.72 | **0.75-0.78** |
| **Precision** | 0.58 | **0.65+** |
| **Recall** | 0.95 | **0.88-0.92** |
| **Test set valid** | ❌ | ✅ |

## Step 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Clone/Update Repository

In [2]:
import os

if not os.path.exists('/content/phase2'):
    !git clone https://github.com/nithin12342/phase2.git /content/phase2
    print("✅ Repository cloned successfully!")
else:
    %cd /content/phase2
    !git fetch origin
    !git reset --hard origin/main
    %cd /content
    print("✅ Repository updated to latest!")

Cloning into '/content/phase2'...
remote: Enumerating objects: 1967, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 1967 (delta 58), reused 113 (delta 41), pack-reused 1831 (from 2)
Receiving objects: 100% (1967/1967), 19.77 MiB | 16.14 MiB/s, done.
Resolving deltas: 100% (1040/1040), done.
✅ Repository cloned successfully!


## Step 3: Install Dependencies

In [3]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers h5py pandas scikit-learn tqdm --quiet
print("✅ Dependencies installed!")

✅ Dependencies installed!


## Step 4: Configure Paths & Verify Phase 6 Checkpoints

In [4]:
# ============================================
# Phase 7 Configuration - Fine-tuning from Phase 6
# ============================================

OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
DATA_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output"
LABELS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv"

# Phase 6 checkpoints to resume from
PHASE6_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase6"

# Phase 7 output directory
OUT_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase7"

# Create output directory
!mkdir -p {OUT_DIR}

# Verify paths exist
import os
import glob

print(f"📁 OS_PATH exists: {os.path.exists(OS_PATH)}")
print(f"📁 DATA_DIR exists: {os.path.exists(DATA_DIR)}")
print(f"📁 LABELS exists: {os.path.exists(LABELS)}")
print(f"📁 PHASE6_DIR exists: {os.path.exists(PHASE6_DIR)}")
print(f"📁 OUT_DIR exists: {os.path.exists(OUT_DIR)}")

# List Phase 6 checkpoints
print("\n📦 Phase 6 Checkpoints:")
phase6_checkpoints = sorted(glob.glob(f"{PHASE6_DIR}/*.pt"))
for ckpt in phase6_checkpoints:
    size_mb = os.path.getsize(ckpt) / (1024*1024)
    print(f"   {os.path.basename(ckpt)} ({size_mb:.1f} MB)")

if not phase6_checkpoints:
    print("   ⚠️ No Phase 6 checkpoints found! Will train from scratch.")

📁 OS_PATH exists: True
📁 DATA_DIR exists: True
📁 LABELS exists: True
📁 PHASE6_DIR exists: True
📁 OUT_DIR exists: True

📦 Phase 6 Checkpoints:
   h5_omnifusion_medium_fold0_best.pt (128.0 MB)
   h5_omnifusion_medium_fold1_best.pt (128.0 MB)
   h5_omnifusion_medium_fold2_best.pt (128.0 MB)
   h5_omnifusion_medium_fold3_best.pt (128.0 MB)
   h5_omnifusion_medium_fold4_best.pt (128.0 MB)


## Step 5: Fine-Tune All 5 Folds from Phase 6 Checkpoints

In [5]:
# ============================================
# Phase 7 Fine-Tuning from Phase 6 Checkpoints
# ============================================
# Key: Uses --resume to load Phase 6 weights and continue training
# with the new Phase 7 fixes (stratified split, label smoothing 0.1)

import time
import os
import glob

# Detect tier from Phase 6 checkpoints
phase6_checkpoints = sorted(glob.glob(f"{PHASE6_DIR}/*.pt"))

# Determine tier from checkpoint name
if phase6_checkpoints:
    sample_ckpt = os.path.basename(phase6_checkpoints[0])
    if '_micro_' in sample_ckpt:
        TIER = 'micro'
    elif '_nano_' in sample_ckpt:
        TIER = 'nano'
    elif '_medium_' in sample_ckpt:
        TIER = 'medium'
    else:
        TIER = 'micro'  # Default
    print(f"🔍 Detected tier from Phase 6: {TIER}")
else:
    TIER = 'micro'
    print(f"⚠️ No Phase 6 checkpoints, using default tier: {TIER}")

total_start = time.time()

for fold in range(5):
    print(f"\n{'='*60}")
    print(f"🚀 PHASE 7 - FOLD {fold}/4 (Fine-tuning from Phase 6)")
    print(f"{'='*60}\n")

    # Find Phase 6 checkpoint for this fold
    phase6_ckpt = f"{PHASE6_DIR}/h5_omnifusion_{TIER}_fold{fold}_best.pt"

    fold_start = time.time()

    if os.path.exists(phase6_ckpt):
        print(f"📥 Resuming from: {phase6_ckpt}")
        !PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/train.py \
            --data_dir {DATA_DIR} \
            --labels_csv {LABELS} \
            --output_dir {OUT_DIR} \
            --tier {TIER} \
            --epochs 15 \
            --fold_idx {fold} \
            --resume {phase6_ckpt}
    else:
        print(f"⚠️ Phase 6 checkpoint not found: {phase6_ckpt}")
        print(f"   Training fold {fold} from scratch...")
        !PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/train.py \
            --data_dir {DATA_DIR} \
            --labels_csv {LABELS} \
            --output_dir {OUT_DIR} \
            --tier {TIER} \
            --epochs 15 \
            --fold_idx {fold}

    fold_time = time.time() - fold_start
    print(f"\n⏱️ Fold {fold} completed in {fold_time/60:.1f} minutes")

total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"✅ ALL 5 FOLDS COMPLETED!")
print(f"⏱️ Total training time: {total_time/60:.1f} minutes")
print(f"{'='*60}")

🔍 Detected tier from Phase 6: medium

🚀 PHASE 7 - FOLD 0/4 (Fine-tuning from Phase 6)

📥 Resuming from: /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase6/h5_omnifusion_medium_fold0_best.pt
Utils loaded. Device: cuda
Available: librosa=True, opensmile=False, cv2=True, transformers=True
ModelLoader initialized. Pretrained path: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models
Using device: cuda

Configuration:
  Tier: medium
  Dimension: 256
  Params: Audio=facebook/wav2vec2-large-xlsr-53, Text=mental/mental-roberta-base
  Folds: 5 (Current: 0)
  Labels CSV: /content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv

Initializing model...
  Total parameters: 11.98M

Loading data from /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output...
  Using labels from /content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv
[H5Dataset] Loaded 358 labels from /content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv
[H5Dataset] Found 358 H5 files acros

## Step 6: Generate Ensemble Predictions

In [6]:
# ============================================
# Generate Ensemble Predictions
# ============================================

print("🔮 Generating ensemble predictions...\n")

!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/ensemble_predict.py \
    --checkpoints {OUT_DIR} \
    --input {DATA_DIR} \
    --tier {TIER} \
    --output "/content/phase7_results.csv"

print("\n✅ Ensemble predictions saved to /content/phase7_results.csv")

🔮 Generating ensemble predictions...

Utils loaded. Device: cuda
Available: librosa=True, opensmile=False, cv2=True, transformers=True
ModelLoader initialized. Pretrained path: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models
Using device: cuda
Traceback (most recent call last):
  File "/content/phase2/ml_pipeline/h5_omnifusion/scripts/ensemble_predict.py", line 182, in <module>
    main()
  File "/content/phase2/ml_pipeline/h5_omnifusion/scripts/ensemble_predict.py", line 151, in main
    predictor = H5EnsemblePredictor(checkpoint_list, args.tier)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/phase2/ml_pipeline/h5_omnifusion/scripts/ensemble_predict.py", line 44, in __init__
    model.load_state_dict(state_dict)
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 2629, in load_state_dict
    raise RuntimeError(
RuntimeError: Error(s) in loading state_dict for H5OmniFusion:
	Missing key(s) in state_dict: "modali

## Step 7: Evaluate Ensemble Results

In [11]:
# ============================================
# DEBUG & FIX: Check checkpoints and run ensemble
# ============================================
import os
import glob

print("🔍 DEBUGGING Phase 7...\n")

# Check what checkpoints exist
print("📦 Phase 7 Output Directory:")
print(f"   Path: {OUT_DIR}")
print(f"   Exists: {os.path.exists(OUT_DIR)}")

p7_ckpts = glob.glob(f"{OUT_DIR}/*.pt")
print(f"\n📦 Phase 7 Checkpoints ({len(p7_ckpts)} found):")
for c in p7_ckpts:
    print(f"   {os.path.basename(c)} ({os.path.getsize(c)/1024/1024:.1f} MB)")

# If no Phase 7, try Phase 6
if not p7_ckpts:
    print("\n⚠️ No Phase 7 checkpoints! Checking Phase 6...")
    p6_ckpts = glob.glob(f"{PHASE6_DIR}/*.pt")
    print(f"\n📦 Phase 6 Checkpoints ({len(p6_ckpts)} found):")
    for c in p6_ckpts:
        print(f"   {os.path.basename(c)} ({os.path.getsize(c)/1024/1024:.1f} MB)")

    if p6_ckpts:
        print("\n🔄 Using Phase 6 checkpoints for ensemble...")
        CKPT_DIR = PHASE6_DIR
        # Detect tier
        if '_micro_' in p6_ckpts[0]:
            TIER = 'micro'
        elif '_nano_' in p6_ckpts[0]:
            TIER = 'nano'
        else:
            TIER = 'medium'
    else:
        print("❌ No checkpoints found anywhere!")
        CKPT_DIR = None
else:
    CKPT_DIR = OUT_DIR
    # Detect tier
    if '_micro_' in p7_ckpts[0]:
        TIER = 'micro'
    elif '_nano_' in p7_ckpts[0]:
        TIER = 'nano'
    else:
        TIER = 'medium'

# Run ensemble if we have checkpoints
if CKPT_DIR:
    print(f"\n🚀 Running ensemble with tier={TIER} from {CKPT_DIR}...")
    !PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/ensemble_predict.py \
        --checkpoints {CKPT_DIR} \
        --input {DATA_DIR} \
        --tier {TIER} \
        --output "/content/phase7_results.csv"

    # Check if it worked
    if os.path.exists("/content/phase7_results.csv"):
        print("\n✅ SUCCESS! Results file created.")
        !cp /content/phase7_results.csv /content/final_ensemble_results.csv
        !PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/evaluate_ensemble.py
    else:
        print("\n❌ Ensemble prediction still failed. Check error above.")

🔍 DEBUGGING Phase 7...

📦 Phase 7 Output Directory:
   Path: /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase7
   Exists: True

📦 Phase 7 Checkpoints (6 found):
   h5_omnifusion_micro_fold0_best.pt (11.4 MB)
   h5_omnifusion_micro_fold1_best.pt (11.4 MB)
   h5_omnifusion_micro_fold2_best.pt (11.4 MB)
   h5_omnifusion_micro_fold3_best.pt (11.4 MB)
   h5_omnifusion_micro_fold4_best.pt (11.4 MB)
   h5_omnifusion_medium_fold3_best.pt (128.0 MB)

🚀 Running ensemble with tier=micro from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase7...
Utils loaded. Device: cuda
Available: librosa=True, opensmile=False, cv2=True, transformers=True
ModelLoader initialized. Pretrained path: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models
Using device: cuda
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase7/h5_omnifusion_micro_fold0_best.pt
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase7/h5_omnifusion_micro_fold1_best

## Step 8: Review Results

In [12]:
import os
import glob

# Check Phase 7 checkpoints
print("📦 Phase 7 Checkpoints:")
p7_ckpts = glob.glob(f"{OUT_DIR}/*.pt")
for c in p7_ckpts:
    print(f"   {os.path.basename(c)} ({os.path.getsize(c)/1024/1024:.1f} MB)")

if not p7_ckpts:
    print("   ⚠️ No checkpoints found!")

📦 Phase 7 Checkpoints:
   h5_omnifusion_micro_fold0_best.pt (11.4 MB)
   h5_omnifusion_micro_fold1_best.pt (11.4 MB)
   h5_omnifusion_micro_fold2_best.pt (11.4 MB)
   h5_omnifusion_micro_fold3_best.pt (11.4 MB)
   h5_omnifusion_micro_fold4_best.pt (11.4 MB)
   h5_omnifusion_medium_fold3_best.pt (128.0 MB)


## 📊 Phase 7 Complete!

Compare your results with Phase 6:

| Metric | Phase 6 | Phase 7 Target | Your Result |
|--------|---------|----------------|-------------|
| **Ensemble F1** | 0.72 | **0.75-0.78** | ? |
| **Precision** | 0.58 | **0.65+** | ? |
| **Recall** | 0.95 | **0.88-0.92** | ? |
| **Test F1** | 0.0 (broken) | **> 0** | ? |

### Key Improvements Made:
1. ✅ Fine-tuned from Phase 6 checkpoints
2. ✅ Stratified test split (no more 0-depressed test sets)
3. ✅ Label smoothing 0.1 (was 0.02)
4. ✅ focal_alpha 0.75 (was 0.65)
5. ✅ mixup_alpha 0.3 (was 0.4)
6. ✅ audio_dropout 0.20 (was 0.35)